# 05 - K-Means en Pipelines: Ingenieria de Caracteristicas, Cuantizacion y Aprendizaje Semi-Supervisado

## Curso de Machine Learning: Modelos No Supervisados

En la ingenieria de Machine Learning moderna, K-Means es un **bloque constructivo polivalente** que se integra directamente dentro de arquitecturas supervisadas y de compresion de informacion.

En este cuaderno exploraremos cuatro aplicaciones avanzadas de alto valor practico:
1. **Ingenieria de Caracteristicas Basada en Distancia Espacial (`kmeans.transform`)**: Como proyectar datos no lineales en un espacio de distancias a centroides para que un modelo lineal simple aprenda fronteras curvas.
2. **Aprendizaje Semi-Supervisado y Etiquetado Activo (*Active Clustering*)**: Como entrenar un clasificador de alta precision cuando solo tenemos presupuesto para etiquetar el $3.7\%$ de nuestro dataset, eligiendo los prototipos de K-Means y propagando sus etiquetas.
3. **Cuantizacion Vectorial (Vector Quantization)**: Reduccion del espacio de estados y compresion con perdida controlada.
4. **Deteccion de Anomalias por Distancia al Centroide mas Cercano**: Identificacion de registros atipicos mediante el percentil extremo de reconstruccion.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import load_digits, load_wine, make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

# Configuracion visual
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
np.random.seed(42)


## 1. El Metodo `.transform(X)`: Proyeccion al Espacio Cluster-Distancia

Dada una matriz $X \in \mathbb{R}^{n \times p}$, el metodo `kmeans.transform(X)` devuelve una nueva matriz continua $D \in \mathbb{R}^{n \times K}$ donde cada columna $k$ contiene la **distancia euclidiana exacta de la observacion $x_i$ al centroide $\mu_k$**:

$$\Phi(x_i) = \Big[ \Vert{}x_i - \mu_1\Vert{}_2, \; \Vert{}x_i - \mu_2\Vert{}_2, \; \dots, \; \Vert{}x_i - \mu_K\Vert{}_2 \Big] \in \mathbb{R}^K$$


In [ ]:
# Demostracion: Regresion Logistica Lineal vs. Pipeline (KMeans Transformer + Regresion Logistica) en Datos No Lineales
X_nl, y_nl = make_moons(n_samples=600, noise=0.22, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_nl, y_nl, test_size=0.25, random_state=42)

pipe_lineal = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(random_state=42))
]).fit(X_tr, y_tr)

pipe_kmeans_feat = Pipeline([
    ('scaler', StandardScaler()),
    ('kmeans_features', KMeans(n_clusters=15, n_init=10, random_state=42)),
    ('clf', LogisticRegression(C=10.0, max_iter=1000, random_state=42))
]).fit(X_tr, y_tr)

acc_lin = pipe_lineal.score(X_te, y_te)
acc_km = pipe_kmeans_feat.score(X_te, y_te)

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))
xx, yy = np.meshgrid(np.linspace(-2.0, 3.0, 300), np.linspace(-1.5, 2.0, 300))
grid = np.c_[xx.ravel(), yy.ravel()]

for ax, model, titulo in zip(
    axes, 
    [pipe_lineal, pipe_kmeans_feat],
    [f'Regresion Logistica Directa\n(Accuracy Test = {acc_lin*100:.1f}%)',
     f'Pipeline: KMeans(K=15).transform() + LogReg\n(Accuracy Test = {acc_km*100:.1f}%)']
):
    Z = model.predict_proba(grid)[:, 1].reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=20, cmap='RdBu', alpha=0.6)
    ax.contour(xx, yy, Z, levels=[0.5], colors='k', linewidths=2)
    ax.scatter(X_te[:, 0], X_te[:, 1], c=y_te, cmap='RdBu', s=35, edgecolor='k')
    ax.set_title(titulo, fontsize=12, fontweight='bold')
    ax.set_xlabel('$x_1$')
    ax.set_ylabel('$x_2$')

plt.suptitle('Ingenieria de Caracteristicas con K-Means: De Frontera Lineal a Frontera No Lineal', 
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


## 2. Aprendizaje Semi-Supervisado Guiado por K-Means (*Active Clustering*)

Cuando solo tenemos presupuesto para etiquetar manualmente $k = 50$ muestras de un dataset de $1,347$ observaciones, seleccionar los **50 prototipos mas cercanos a los centroides de K-Means** y propagar sus etiquetas al resto de cada cluster supera drasticamente al muestreo aleatorio.


In [ ]:
digits = load_digits()
X_train_ssl, X_test_ssl, y_train_ssl, y_test_ssl = train_test_split(
    digits.data, digits.target, test_size=0.25, random_state=42, stratify=digits.target
)

k_presupuesto = 50

# 1. Estrategia A: 50 muestras elegidas al azar
clf_random = LogisticRegression(max_iter=2000, random_state=42)
clf_random.fit(X_train_ssl[:k_presupuesto], y_train_ssl[:k_presupuesto])
acc_random = clf_random.score(X_test_ssl, y_test_ssl)

# 2. Estrategia B: 50 Prototipos Representativos seleccionados con K-Means
km_ssl = KMeans(n_clusters=k_presupuesto, n_init=10, random_state=42)
X_dist_clusters = km_ssl.fit_transform(X_train_ssl)

idx_prototipos = np.argmin(X_dist_clusters, axis=0)
X_prototipos = X_train_ssl[idx_prototipos]
y_prototipos_humano = y_train_ssl[idx_prototipos]

clf_prototipos = LogisticRegression(max_iter=2000, random_state=42)
clf_prototipos.fit(X_prototipos, y_prototipos_humano)
acc_prototipos = clf_prototipos.score(X_test_ssl, y_test_ssl)

# 3. Estrategia C: Propagacion de la etiqueta del prototipo a los miembros del cluster (85% mas cercanos)
y_propagado = np.empty(len(X_train_ssl), dtype=int)
for k in range(k_presupuesto):
    y_propagado[km_ssl.labels_ == k] = y_prototipos_humano[k]

dist_al_propio_centroide = X_dist_clusters[np.arange(len(X_train_ssl)), km_ssl.labels_]
umbral_confianza = np.percentile(dist_al_propio_centroide, 85)
mask_confiables = (dist_al_propio_centroide <= umbral_confianza)

clf_propagado = LogisticRegression(max_iter=2000, random_state=42)
clf_propagado.fit(X_train_ssl[mask_confiables], y_propagado[mask_confiables])
acc_propagado = clf_propagado.score(X_test_ssl, y_test_ssl)

print("=" * 80)
print(f"IMPACTO DEL APRENDIZAJE SEMI-SUPERVISADO CON K-MEANS (Presupuesto = {k_presupuesto} etiquetas)")
print("=" * 80)
print(f"  1. Entrenamiento con 50 muestras aleatorias:              Accuracy Test = {acc_random*100:.2f}%")
print(f"  2. Entrenamiento con los 50 prototipos de K-Means:        Accuracy Test = {acc_prototipos*100:.2f}%")
print(f"  3. Propagacion de etiquetas a los clusters (85% nucleo):  Accuracy Test = {acc_propagado*100:.2f}%")
print("=" * 80)


## 3. Cuantizacion Vectorial y Deteccion de Anomalias por Distancia al Centroide

Al calcular la distancia minima de cada punto a su centroide mas cercano $e_i = \min_k \Vert{}x_i - \mu_k\Vert{}_2$, podemos tanto comprimir senales continuas como aislar el percentil extremo ($95\%$) de observaciones mas anomalas.


In [ ]:
wine = load_wine(as_frame=True)
X_wine_scaled = StandardScaler().fit_transform(wine.data)

km_anom = KMeans(n_clusters=3, n_init=15, random_state=42)
dist_matriz = km_anom.fit_transform(X_wine_scaled)
dist_minima = np.min(dist_matriz, axis=1)

umbral_95 = np.percentile(dist_minima, 95)
es_anomalia = (dist_minima > umbral_95)

plt.figure(figsize=(11, 4.8))
plt.hist(dist_minima, bins=25, color='#2b5c8f', edgecolor='k', alpha=0.8, label='Distribucion de Distancias al Centroide')
plt.axvline(umbral_95, color='#d62728', linestyle='--', lw=2.5, label=f'Umbral Percentil 95% (d = {umbral_95:.2f})')
plt.xlabel('Distancia Euclidiana al Centroide Asignado (Espacio Z)', fontsize=11, fontweight='bold')
plt.ylabel('Frecuencia de Muestras', fontsize=11, fontweight='bold')
plt.title(f'Deteccion de Anomalias con K-Means: {es_anomalia.sum()} Vinos Detectados en la Cola Extrema', 
          fontsize=12, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()
